##ΜΙΝIPJS-156

In [0]:
env="p"

In [0]:
spark.sql(f"""
ALTER TABLE `{env}-cdh`.primaryactive.portfolio
ADD COLUMN GBVInitializationDate DATE AFTER isCostPlusFlag;
""")

In [0]:
spark.sql(f"""
UPDATE `{env}-cdh`.primaryactive.portfolio
SET GBVInitializationDate='2024-12-31'
WHERE IsHAPSFlag='true';
""")

In [0]:
%sql
INSERT INTO `p-cdh`.derivativeactive.zeroinggbvexposure
SELECT  ExposureKey,
    GBVTriggerEventFlag,
    GBVTriggerEventDate,
    GBVTriggerEventType,
    GBVCappedAmount,
    GBVCappedDate,
    TailExposure as  GBVTailExposure,
    GBVFunctionResult,
    BusinessDate
FROM `p-cdh`.derivativeactive.dexposure
WHERE BusinessDate >= '2024-12-31'
AND GBVCappedDate IS NOT NULL
AND BusinessDate < (
    SELECT MAX(BusinessDate)
    FROM `p-cdh`.derivativeactive.zeroinggbvexposure
);

##D360-935

### Steps 
* Run the following code
* Re-run AuctionRealEstate for all businessdates (first for Novemver and on -- critical to run today --)

In [0]:
spark.sql(f"""
ALTER TABLE `p-cdh`.primaryactive.auctionrealestate
  DROP COLUMN DebtorTaxRegistrationNumber;
""")

In [0]:
spark.sql(f"""
ALTER TABLE `p-cdh`.primaryactive.auctionrealestate
DROP COLUMN DebtorDescription;
""")

In [0]:
spark.sql(f"""
ALTER TABLE `p-cdh`.primaryactive.auctionrealestate
DROP COLUMN AuctionKey;
""")

## D360-2151
### Steps
* Run the following insert statement
* Create a view for derivative.DAuctionRealEstate
* Re - run from November and on (ensure that first the primary.auctionrealestate has completed its rerun)
* Add DAuctionRealEstate in daily flow

In [0]:
%sql
CREATE TABLE `p-cdh`.derivativeactive.dauctionrealestate
AS SELECT * FROM `qa-cdh`.it.dauctionrealestate WHERE businessdate>='2024-01-10'

In [0]:
%sql
CREATE OR REPLACE  VIEW `p-cdh`.derivative.dauctionrealestate
as select * from `p-cdh`.derivativeactive.dauctionrealestate

## DATA2-205

In [0]:
# Delete duplicates from landing.qca_dim_cases_static
spark.sql(f"""
with OrderedTable as (
SELECT distinct CASE_CODE, CASE_LAST_UPD, Cluster_0
, DENSE_RANK() OVER (PARTITION BY  CASE_CODE ORDER BY CASE_LAST_UPD desc) AS row_num
FROM `{env}-cdh`.landing.qca_dim_cases_static 
WHERE Cluster_0 = '2024-03-08'
)


DELETE FROM `{env}-cdh`.landing.qca_dim_cases_static a
WHERE EXISTS (
    SELECT 1
    FROM OrderedTable b
    WHERE b.row_num = 2
      AND b.Cluster_0 = a.Cluster_0
      AND b.CASE_CODE = a.CASE_CODE
      AND b.CASE_LAST_UPD = a.CASE_LAST_UPD
            )
""")

In [0]:

# Delete duplicates from landed.qca_dim_cases_static
spark.sql(f"""
with OrderedTable as (
SELECT distinct CASE_CODE, CASE_LAST_UPD, BusinessDate
, DENSE_RANK() OVER (PARTITION BY  CASE_CODE ORDER BY CASE_LAST_UPD desc) AS row_num
FROM `{env}-cdh`.landed.qca_dim_cases_static 
WHERE BusinessDate = '2024-03-07')

DELETE FROM `{env}-cdh`.landed.qca_dim_cases_static a
WHERE EXISTS (
    SELECT 1
    FROM OrderedTable b
    WHERE b.row_num = 2
      AND b.BusinessDate = a.BusinessDate
      AND b.CASE_CODE = a.CASE_CODE
      AND b.CASE_LAST_UPD = a.CASE_LAST_UPD
            )
""")

Re-run CaseCode, Exposure & Dexposure in prod for BusinessDate = '2024-03-07', once the above code has been successfully executed.

## Refresh views


In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('p-cdh')
      AND table_schema in ('derivative', 'primary')
      and table_type='VIEW'
""")

# display(views_df)

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    # print(table_name)

    full_view_name =f"`{table_catalog}`.{table_schema}.{table_name}"

    describe_full_view_name = spark.sql(f"""
              DESCRIBE EXTENDED {full_view_name}""")
    
    describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

   # print(describe_full_view_name)

    describe_query = spark.sql(f"""
              select data_type--REGEXP_REPLACE(TRIM(data_type), '[^a-zA-Z0-9*<> ]', '') 
              AS cleaned_view_text 
              from describe_full_view_name 
              where col_name ='View Text'
              """)
    
    sql_view_creation = describe_query.collect()[0][0]

    create_or_replace_sql = f"""
          create or replace view {full_view_name} as 
          {sql_view_creation}
          """
    
    spark.sql(create_or_replace_sql)



In [0]:
# Get the schema of the dexposure table
dexposure_table = spark.table(f"`{env}-cdh`.derivative.dexposure")
dexposure_schema_fields = dexposure_table.schema.fields


# Convert the schema fields into the required string format
dexposure_schema = ",\n".join([
    f"{field.name} {field.dataType.simpleString()}"
    for field in dexposure_schema_fields
])


spark.sql(f'''
CREATE OR REPLACE FUNCTION `{env}-cdh`.derivative.get_all_exposures_by_business_date(
    business_date Date
)
RETURNS TABLE 
( 
{dexposure_schema}
)
RETURN

SELECT B.*
FROM(
        SELECT  ExposureKey, MAX(BusinessDate) BusinessDate
        FROM `{env}-cdh`.derivative.dexposure
        WHERE BusinessDate <= business_date
        GROUP BY ExposureKey
) A
INNER JOIN   `{env}-cdh`.derivative.dexposure B
on a.ExposureKey = b.ExposureKey
and a.BusinessDate = b.BusinessDate
AND B.IsActiveFlagInt = 0

UNION

SELECT * FROM `{env}-cdh`.derivative.dexposure
WHERE BusinessDate = business_date
AND IsActiveFlagInt = 1
''')